<a href="https://colab.research.google.com/github/FDM20000/SIAFAX/blob/main/SIAFAX_Frames.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SIAFAX 2

1 - Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2 - Definir caminhos do projeto

In [ ]:
import os

base_dir = "/content/drive/MyDrive/projeto_rodovia"

video_path  = f"{base_dir}/MVP389c.MOV"
gpx_path    = f"{base_dir}/TRILHA_MVP389c.gpx"
frames_dir  = f"{base_dir}/frames"
yolo_dir    = f"{base_dir}/yolo_dataset"
images_dir  = f"{yolo_dir}/images"
labels_dir  = f"{yolo_dir}/labels"

os.makedirs(frames_dir, exist_ok=True)
os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

video_path, gpx_path


('/content/drive/MyDrive/projeto_rodovia/MVP389c.MOV',
 '/content/drive/MyDrive/projeto_rodovia/TRILHA_MVP389c.gpx')

In [ ]:
# Verificar se o vídeo existe

import os
os.path.exists(video_path)


True

3 - Instalar ffmpeg

In [ ]:
!apt-get install -y ffmpeg


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


4 - Gerar Frames

In [ ]:
# Limpar frames

!rm -f {frames_dir}/frame_*.jpg


In [ ]:
# Extrair 1 frame por segundo

!ffmpeg -i "{video_path}" -vf "fps=1" "{frames_dir}/frame_%06d.jpg"


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

5 - Verificar se os frames foram criados

In [ ]:
import glob

frames = sorted(glob.glob(f"{frames_dir}/frame_*.jpg"))
len(frames), frames[:5]


(786,
 ['/content/drive/MyDrive/projeto_rodovia/frames/frame_000001.jpg',
  '/content/drive/MyDrive/projeto_rodovia/frames/frame_000002.jpg',
  '/content/drive/MyDrive/projeto_rodovia/frames/frame_000003.jpg',
  '/content/drive/MyDrive/projeto_rodovia/frames/frame_000004.jpg',
  '/content/drive/MyDrive/projeto_rodovia/frames/frame_000005.jpg'])

6 - Copiar frames para YOLO

In [ ]:
# Limpar pasta YOLO/images

!rm -f {images_dir}/frame_*.jpg


In [ ]:
# Copiar frames

import shutil

for f in frames:
    shutil.copy(f, images_dir)


In [ ]:
# Verificar se foram copiados

yolo_frames = sorted(glob.glob(f"{images_dir}/frame_*.jpg"))
len(yolo_frames), yolo_frames[:5]


(786,
 ['/content/drive/MyDrive/projeto_rodovia/yolo_dataset/images/frame_000001.jpg',
  '/content/drive/MyDrive/projeto_rodovia/yolo_dataset/images/frame_000002.jpg',
  '/content/drive/MyDrive/projeto_rodovia/yolo_dataset/images/frame_000003.jpg',
  '/content/drive/MyDrive/projeto_rodovia/yolo_dataset/images/frame_000004.jpg',
  '/content/drive/MyDrive/projeto_rodovia/yolo_dataset/images/frame_000005.jpg'])

7 - Ler GPX

In [ ]:
!pip install gpxpy

import gpxpy

with open(gpx_path, 'r') as f:
    gpx = gpxpy.parse(f)

gps_points = []

for track in gpx.tracks:
    for segment in track.segments:
        for point in segment.points:
            gps_points.append({
                "time": point.time,
                "lat": point.latitude,
                "lon": point.longitude,
                "alt": point.elevation
            })

len(gps_points)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.0 MB/s eta 0:00:00


394

8 - Sincronizar FRAMES ↔ GPS

In [ ]:
# Pegar creation_time do vídeo

import json, subprocess

cmd = [
    "ffprobe", "-v", "quiet",
    "-print_format", "json",
    "-show_format", "-show_streams",
    video_path
]
result = subprocess.run(cmd, capture_output=True, text=True)
info = json.loads(result.stdout)

info["format"]["tags"]["creation_time"]


'2026-05-09T13:32:07.000000Z'

In [ ]:
# Preparar sincronização

from datetime import datetime, timedelta, timezone

creation_time = datetime.fromisoformat("2026-04-18T17:30:50.000000+00:00")

for p in gps_points:
    if p["time"].tzinfo is None:
        p["time"] = p["time"].replace(tzinfo=timezone.utc)


In [ ]:
# Calcular FPS real

video_stream = next(s for s in info["streams"] if s["codec_type"] == "video")
duration_sec = float(info["format"]["duration"])
nb_frames = len(frames)

fps = nb_frames / duration_sec
fps


1.0003330205349674

In [ ]:
# Função para achar ponto GPS mais próximo

def find_nearest_gps(time_target, gps_points):
    best = None
    best_dt = None
    for p in gps_points:
        dt = abs((p["time"] - time_target).total_seconds())
        if best is None or dt < best_dt:
            best = p
            best_dt = dt
    return best


In [ ]:
# Criar tabela frame → GPS

frame_gps = []

for i in range(nb_frames):
    t_frame = creation_time + timedelta(seconds=i / fps)
    p = find_nearest_gps(t_frame, gps_points)

    frame_gps.append({
        "frame_idx": i + 1,
        "frame_file": f"frame_{(i+1):06d}.jpg",
        "time": t_frame,
        "lat": p["lat"],
        "lon": p["lon"],
        "alt": p["alt"],
    })

len(frame_gps)


786

9 - Salvar CSV final

In [ ]:
import csv

csv_path = f"{base_dir}/frames_gps.csv"

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["frame_idx", "frame_file", "time", "lat", "lon", "alt"])
    writer.writeheader()
    for row in frame_gps:
        writer.writerow(row)

csv_path


'/content/drive/MyDrive/projeto_rodovia/frames_gps.csv'

In [ ]:
import pandas as pd
df = pd.read_csv(csv_path)
len(df)


786

In [ ]:
df.head()
df.tail()


,frame_idx,frame_file,time,lat,lon,alt
781,782,frame_000782.jpg,2026-04-18 17:43:50.739998+00:00,-29.913162,-50.274259,14.3
782,783,frame_000783.jpg,2026-04-18 17:43:51.739665+00:00,-29.913162,-50.274259,14.3
783,784,frame_000784.jpg,2026-04-18 17:43:52.739332+00:00,-29.913162,-50.274259,14.3
784,785,frame_000785.jpg,2026-04-18 17:43:53.738999+00:00,-29.913162,-50.274259,14.3
785,786,frame_000786.jpg,2026-04-18 17:43:54.738666+00:00,-29.913162,-50.274259,14.3
